# 04 · Features, task types and model configuration

A `Feature` describes one input column and what (if anything) to predict from it. The model builds
its embeddings, heads and losses from this list.

## The task types

| You have | `Feature(...)` | Output per sample | Loss |
|---|---|---|---|
| categorical input, predict the next token | `Feature("item", "classification", vocab_size=V)` | `V` logits | cross entropy |
| numeric input, predict a value (any sign) | `Feature("price", "regression", is_linear=True)` | 1 value | MSE |
| numeric input, predict a value that is **> 0** | `Feature("gap", "positive_regression", is_linear=True)` | 1 value (always > 0) | MSE |
| numeric input, predict an **interval** | `Feature("price", "quantile", is_linear=True, coverage=(0.5, 0.95))` | median + bounds | pinball |
| numeric input, predict a bucket | `Feature("qty", "classification", is_linear=True, n_classes=3)` | 3 logits | cross entropy |
| input only | `Feature("weekday", "passthrough", vocab_size=8)` | none | none |

- `vocab_size`: number of token ids of a **categorical input**, including PAD (0) and UNK (1).
- `is_linear=True`: a **numeric input**, embedded with `nn.Linear(1, d)`.
- `n_classes`: number of classes of a **classification target**; defaults to `vocab_size`.
- `coverage`: the interval(s) of a **quantile** feature, any value(s) in (0, 1). Default `0.95`.

In [1]:
from next_sequence import Feature, TaskType

features = [
    Feature("item", TaskType.CLASSIFICATION, vocab_size=7),
    Feature("price", "quantile", is_linear=True, coverage=(0.5, 0.95)),   # strings work too
    Feature("gap", "positive_regression", is_linear=True),
    Feature("qty", "classification", is_linear=True, n_classes=3),
    Feature("weekday", "passthrough", vocab_size=8),
]
for f in features:
    print(f.name, "->", f.task.value, f"| quantiles: {f.quantiles}" if f.quantiles else "")

item -> classification 
price -> quantile | quantiles: (0.025, 0.25, 0.5, 0.75, 0.975)
gap -> positive_regression 
qty -> classification 
weekday -> passthrough 


## Mistakes are caught immediately

Invalid combinations raise a `ValueError` that says what to change.

In [2]:
attempts = {
    "categorical input without vocab_size": dict(name="item", task="classification"),
    "numeric classification without n_classes": dict(name="qty", task="classification", is_linear=True),
    "coverage written as a percentage": dict(name="price", task="quantile", is_linear=True, coverage=95),
    "coverage on a regression": dict(name="price", task="regression", is_linear=True, coverage=0.9),
    "a task that doesn't exist": dict(name="price", task="interval", is_linear=True),
    "a dot in the name": dict(name="item.id", task="regression", is_linear=True),
}
for label, kwargs in attempts.items():
    try:
        Feature(**kwargs)
    except ValueError as e:
        print(f"{label:42s} -> {e}")

categorical input without vocab_size       -> item: a categorical input needs vocab_size >= 2 (including PAD). For a numeric input, set is_linear=True.
numeric classification without n_classes   -> qty: classification needs n_classes >= 2, got 0
coverage written as a percentage           -> price: each coverage must be a number strictly between 0 and 1 (e.g. 0.5 or 0.95), got 95
coverage on a regression                   -> price: coverage is only used with task=quantile
a task that doesn't exist                  -> 'interval' is not a valid TaskType
a dot in the name                          -> Feature name must not contain '.', got 'item.id'


## Model configuration

`NextSequenceConfig` holds the architecture. The width of the transformer is `fusion_dim`
(default: `emb_dim`), and `fusion_dim / attn_heads` must be an even integer.
`recent_first=True` (the default) feeds every history newest event first; collates built with
`CustomCollate.from_config` follow it automatically.

In [3]:
from next_sequence import NextSequenceConfig

config = NextSequenceConfig(features=features, max_len=16, emb_dim=32, attn_heads=2, num_layers=2)
{k: v for k, v in config.to_dict().items() if k != "features"}

{'schema_version': 2,
 'max_len': 16,
 'emb_dim': 32,
 'fusion_dim': 32,
 'fusion_strategy': 'sum',
 'rope_strategy': 'interleaved',
 'attn_heads': 2,
 'num_layers': 2,
 'expansion': 2,
 'dropout': 0.1,
 'recent_first': True}

In [4]:
bad_configs = {
    "fusion_dim not divisible by heads": dict(emb_dim=30, attn_heads=4),
    "odd head width": dict(emb_dim=18, attn_heads=2),
    "duplicate feature": dict(features=features + [features[0]]),
    "nothing to predict": dict(features=[Feature("weekday", "passthrough", vocab_size=8)]),
}
for label, kwargs in bad_configs.items():
    try:
        NextSequenceConfig(**{"features": features, "max_len": 16, **kwargs})
    except ValueError as e:
        print(f"{label:36s} -> {e}")

fusion_dim not divisible by heads    -> fusion_dim (30) must be divisible by attn_heads (4)
odd head width                       -> fusion_dim / attn_heads = 9 must be even (RoPE)
duplicate feature                    -> duplicate feature names: ['item']
nothing to predict                   -> at least one feature must be predicted (not PASSTHROUGH)


A config is plain data, so it can be saved as JSON and rebuilt:

In [5]:
import json

as_json = json.dumps(config.to_dict())
assert NextSequenceConfig.from_dict(json.loads(as_json)) == config
print(len(as_json), "characters of JSON, rebuilt identically")

815 characters of JSON, rebuilt identically


## What the model looks like

`NextSequenceModel` is a plain `nn.Module`:
embeddings → fusion → CLS token + transformer encoder → one head per predicted feature.

In [6]:
from next_sequence import NextSequenceModel

model = NextSequenceModel(config)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
model.heads

18,384 parameters


MultiTaskHeads(
  (heads): ModuleDict(
    (item): Linear(in_features=32, out_features=7, bias=True)
    (price): Linear(in_features=32, out_features=5, bias=True)
    (gap): Linear(in_features=32, out_features=1, bias=True)
    (qty): Linear(in_features=32, out_features=3, bias=True)
  )
)

## A forward pass on a fake batch

Categorical inputs are `(B, T)` token ids, numeric inputs `(B, T, 1)`. Outputs are raw: logits for
classification, values for regression, one column per quantile (lowest first) for quantile features.
The passthrough feature has no output.

In [7]:
import torch

B, T = 4, 6
X = {"item": torch.randint(2, 7, (B, T)), "price": torch.randn(B, T, 1), "gap": torch.rand(B, T, 1),
     "qty": torch.randn(B, T, 1), "weekday": torch.randint(1, 8, (B, T))}
mask = torch.zeros(B, T, dtype=torch.bool)
mask[0, 4:] = True                              # the first sequence has 2 padded steps

model.eval()
with torch.no_grad():
    out = model(X, mask)
print({name: tuple(t.shape) for name, t in out.items()})
print("gap is always positive:", bool((out["gap"] > 0).all()))
print("quantile columns never cross:", bool((out["price"][:, 1:] >= out["price"][:, :-1]).all()))

{'item': (4, 7), 'price': (4, 5), 'gap': (4, 1), 'qty': (4, 3)}
gap is always positive: True
quantile columns never cross: True


## Combining features: `sum` vs `concat`

- `sum` adds the feature embeddings: width stays `emb_dim`.
- `concat` stacks them side by side (`n_features * emb_dim`), then projects to `fusion_dim`.

In [8]:
from next_sequence import FusionStrategy

for strategy in FusionStrategy:
    m = NextSequenceModel(NextSequenceConfig(features=features, max_len=16, emb_dim=32,
                                             attn_heads=2, fusion_strategy=strategy))
    print(f"{strategy.value:7s} {sum(p.numel() for p in m.parameters()):>8,} parameters | fusion: {m.backbone.fusion}")

sum       18,384 parameters | fusion: SumFusion(
  (proj): Identity()
)
concat    23,536 parameters | fusion: ConcatFusion(
  (proj): Linear(in_features=160, out_features=32, bias=True)
)
